In [ ]:
# ========== Imports ==========
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as ipw
from IPython.display import display
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
!pip install -q plotly seaborn ipywidgets --upgrade


In [ ]:
# ========== Load and Filter Data ==========
url = "https://data.gov.il/api/3/action/datastore_search?resource_id=053cea08-09bc-40ec-8f7a-156f0677aff3&limit=10000"
raw_data = requests.get(url).json()
records = raw_data['result']['records']
df = pd.DataFrame(records)

In [ ]:
# ========== Sunburst Data Preparation ==========
sunburst_data = df.dropna(subset=['tozeret_nm', 'kinuy_mishari', 'ramat_gimur']).copy()
sunburst_data['node_id'] = sunburst_data['tozeret_nm'] + " - " + sunburst_data['kinuy_mishari'] + " - " + sunburst_data['ramat_gimur']
sunburst_data['node_parent'] = sunburst_data['tozeret_nm'] + " - " + sunburst_data['kinuy_mishari']

root = sunburst_data[['tozeret_nm']].drop_duplicates().rename(columns={'tozeret_nm': 'label'})
root['parent'] = ""
root['id'] = root['label']

mid = sunburst_data[['tozeret_nm', 'kinuy_mishari']].drop_duplicates()
mid['label'] = mid['kinuy_mishari']
mid['parent'] = mid['tozeret_nm']
mid['id'] = mid['tozeret_nm'] + " - " + mid['kinuy_mishari']

leaf = sunburst_data[['node_id', 'ramat_gimur', 'node_parent']].rename(
    columns={'node_id': 'id', 'ramat_gimur': 'label', 'node_parent': 'parent'}
)

sunburst_all = pd.concat([root[['id', 'label', 'parent']],
                           mid[['id', 'label', 'parent']],
                           leaf], ignore_index=True)

In [ ]:
# ========== Heatmap Preparation ==========
heatmap_ready = df.dropna(subset=['tozeret_nm', 'shnat_yitzur'])
heat_pivot = heatmap_ready.pivot_table(index='tozeret_nm', columns='shnat_yitzur', aggfunc='size', fill_value=0)

In [ ]:
# ========== Create Combined Plot ==========
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'heatmap'}]],
    column_widths=[0.45, 0.55],
    subplot_titles=["Sunburst Diagram", "Production Distribution"]
)

# Add Sunburst
fig.add_trace(
    go.Sunburst(
        ids=sunburst_all['id'],
        labels=sunburst_all['label'],
        parents=sunburst_all['parent'],
        maxdepth=3
    ),
    row=1, col=1
)

# Add Heatmap
fig.add_trace(
    go.Heatmap(
        z=heat_pivot.values,
        x=heat_pivot.columns.astype(str),
        y=heat_pivot.index,
        colorscale='Viridis'
    ),
    row=1, col=2
)

fig.update_layout(
    title="Vehicles Visualization Dashboard",
    height=700,
    margin=dict(t=40, l=0, r=0, b=0)
)

fig.show()

In [ ]:
# ========== Tabbed Dashboard ==========
# Tab 1 - Overview
overview_tab = ipw.Output()
with overview_tab:
    display(df.describe())
    plt.figure(figsize=(10, 4))
    sns.histplot(df['shnat_yitzur'].dropna(), bins=25, kde=True, color='skyblue')
    plt.title("Distribiotion of Age")
    plt.xlabel("SHnat yestzor")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# Tab 2 - Raw Data
raw_tab = ipw.Output()
with raw_tab:
    display(df.head(15))

# Tab 3 - Yearly Count Bar Chart
bar_tab = ipw.Output()
with bar_tab:
    yearly = df['shnat_yitzur'].value_counts().sort_index()
    plt.figure(figsize=(12, 5))
    plt.bar(yearly.index.astype(int), yearly.values, color='orange')
    plt.title("Vehicles by Year")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Tab 4 - Interactive Explorer
explore_tab = ipw.Output()
brand_dropdown = ipw.Dropdown(
    options=[''] + sorted(df['tozeret_nm'].dropna().unique()),
    description='Brand:',
    style={'description_width': 'initial'}
)
model_dropdown = ipw.Dropdown(options=[''], description='Model:')
result_output = ipw.Output()

def refresh_models(change):
    selected = change['new']
    if selected:
        models = df[df['tozeret_nm'] == selected]['kinuy_mishari'].dropna().unique()
        model_dropdown.options = [''] + sorted(models)
    else:
        model_dropdown.options = ['']

def display_info(change=None):
    result_output.clear_output()
    selected_brand = brand_dropdown.value
    selected_model = model_dropdown.value
    if selected_brand and selected_model:
        filt = df[(df['tozeret_nm'] == selected_brand) & (df['kinuy_mishari'] == selected_model)]
        with result_output:
            print(f"Total Records: {len(filt)}")
            unique_trims = filt['ramat_gimur'].dropna().unique()
            print(f"Trims: {', '.join(unique_trims)}" if len(unique_trims) > 0 else "No trims found.")
    else:
        with result_output:
            print("Choose brand and model to see details.")

brand_dropdown.observe(refresh_models, names='value')
model_dropdown.observe(display_info, names='value')

with explore_tab:
    display(ipw.VBox([brand_dropdown, model_dropdown, result_output]))


In [ ]:
# ========== Final Tabs View ==========
tabs = ipw.Tab(children=[overview_tab, raw_tab, bar_tab, explore_tab])
tab_titles = ["Overview", "Raw", "Year Stats", "Trim Finder"]

for idx, title in enumerate(tab_titles):
    tabs.set_title(idx, title)

display(tabs)